In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")


# Phase 4 — Feature Transformation & Scaling

**Project:** Wholesale Customer Segmentation

Phase 4 - Feature Transformation & Scaling
Wholesale Customers Clustering Analysis

Steps covered (per implementation plan):
 9.  Transform features when justified (log-transform)
 10. Standardize the features

Depends on: phase1_setup.py (df_raw), phase2_eda.py (SPEND_COLS, select_clustering_features),
            phase3_skew_outliers_multicollinearity.py (skewness findings that justify this transform)
Outputs: PNG figures saved to ./figures/, transformed_features.csv, printed before/after diagnostics

### How to use this notebook
Run cells from top to bottom. Keep the project files in the same folder as this notebook. Phases 1–4 create the data preparation artifacts used by later phases; Phases 5–9 read those artifacts; Phase 10 assembles the final report.

In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler



# Recreate upstream constants/data locally so this notebook is standalone.
DATA_PATH = "data/raw/wholesale_customers.csv"
df_raw = pd.read_csv(DATA_PATH)
SPEND_COLS = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
FIG_DIR = "figures"
import os
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_style("whitegrid")

## SECTION 1: Prepare Working Copy (preserve df_raw)

In [ ]:
# SECTION 1: Prepare Working Copy (preserve df_raw)
# ===========================================================================
def get_feature_matrix() -> pd.DataFrame:
    """Retrieve the 6-column clustering feature matrix as a fresh copy.
    df_raw itself is never modified (Phase 1 rule). Selection logic
    (6 spend columns, Channel/Region excluded) matches Phase 2's decision;
    reselected directly here (no verbose re-print) to keep this phase's
    own section numbering clean."""
    print("=" * 70)
    print("SECTION 1: PREPARE WORKING COPY")
    print("=" * 70)

    features_df = df_raw[SPEND_COLS].copy()
    print(f"Working feature matrix shape: {features_df.shape}")
    print(f"Columns: {list(features_df.columns)}")
    print("[OK] df_raw untouched; all transforms below operate on this copy.")
    return features_df

## SECTION 2: Apply Log1p Transform (Step 9)

In [ ]:
# SECTION 2: Apply Log1p Transform (Step 9)
# ===========================================================================
def apply_log_transform(features_df: pd.DataFrame) -> pd.DataFrame:
    """Apply log1p to compress extreme spend values and reduce right-skew while
    retaining all customers. The transform is evaluated by comparing skewness
    before and after transformation."""
    print("\n" + "=" * 70)
    print("SECTION 2: APPLY LOG1P TRANSFORM")
    print("=" * 70)

    log_df = features_df.copy()
    for col in SPEND_COLS:
        log_df[col] = np.log1p(features_df[col])

    print("Applied log1p(x) = log(1 + x) to all 6 spend columns.")
    print(f"Min value pre-transform across features: {features_df[SPEND_COLS].min().min()} "
          "(always > 0 in this dataset, so log1p is a safe, standard choice).")
    print("\nPost-log1p summary statistics:")
    print(log_df.describe().round(2).to_string())
    return log_df

## SECTION 3: Verify Skewness Reduction (supports Step 9 justification)

In [ ]:
# SECTION 3: Verify Skewness Reduction (supports Step 9 justification)
# ===========================================================================
def verify_skew_reduction(features_df: pd.DataFrame, log_df: pd.DataFrame) -> pd.DataFrame:
    """Compare skewness before vs. after log1p to confirm the transform
    achieved its purpose (referenced against Phase 3's skewness findings)."""
    print("\n" + "=" * 70)
    print("SECTION 3: VERIFY SKEWNESS REDUCTION (BEFORE VS. AFTER)")
    print("=" * 70)

    skew_before = features_df[SPEND_COLS].skew()
    skew_after = log_df[SPEND_COLS].skew()
    comparison = pd.DataFrame({
        "skew_before_log": skew_before,
        "skew_after_log": skew_after,
        "improvement": skew_before - skew_after,
    }).round(2)
    print(comparison.to_string())

    still_high = comparison[comparison["skew_after_log"].abs() > 1]
    if len(still_high) > 0:
        print(f"\n[NOTE] Features still moderately/highly skewed after log1p: "
              f"{list(still_high.index)}. This is expected and acceptable — "
              "log1p reduces skew substantially but does not force perfect "
              "symmetry; standardization (Section 4) will further equalize scale.")
    else:
        print("\n[OK] All features now within acceptable skew range after log1p.")
    return comparison

## SECTION 4: Visualize Before/After Distributions

In [ ]:
# SECTION 4: Visualize Before/After Distributions
# ===========================================================================
def plot_before_after_histograms(features_df: pd.DataFrame, log_df: pd.DataFrame) -> None:
    """Side-by-side histograms showing the effect of log1p per feature."""
    print("\n" + "=" * 70)
    print("SECTION 4: VISUALIZE BEFORE/AFTER DISTRIBUTIONS")
    print("=" * 70)

    # Compact 3x4 layout: each feature occupies one adjacent Before/After pair.
    # This keeps the figure readable and prevents it from being split across Word pages.
    fig, axes = plt.subplots(3, 4, figsize=(15, 10))
    axes = axes.reshape(3, 4)
    for i, col in enumerate(SPEND_COLS):
        row = i // 2
        col_pair = (i % 2) * 2
        sns.histplot(features_df[col], bins=30, kde=True, ax=axes[row, col_pair], color="steelblue")
        axes[row, col_pair].set_title(f"{col} — Before (skew={features_df[col].skew():.2f})", fontsize=10)
        axes[row, col_pair].set_xlabel("Annual spend")

        sns.histplot(log_df[col], bins=30, kde=True, ax=axes[row, col_pair + 1], color="seagreen")
        axes[row, col_pair + 1].set_title(f"{col} — After log1p (skew={log_df[col].skew():.2f})", fontsize=10)
        axes[row, col_pair + 1].set_xlabel("log1p(annual spend)")

    fig.suptitle("Effect of log1p Transform on Spend Distributions", fontsize=14, y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(f"{FIG_DIR}/07_log_transform_before_after.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {FIG_DIR}/07_log_transform_before_after.png")

## SECTION 5: Standardize Features (Step 10)

In [ ]:
# SECTION 5: Standardize Features (Step 10)
# ===========================================================================
def standardize_features(log_df: pd.DataFrame) -> tuple[pd.DataFrame, StandardScaler]:
    """Apply StandardScaler (zero mean, unit variance) to the log-transformed
    features. Required because K-Means uses Euclidean distance and raw scales
    differ by orders of magnitude across categories."""
    print("\n" + "=" * 70)
    print("SECTION 5: STANDARDIZE FEATURES (StandardScaler)")
    print("=" * 70)

    scaler = StandardScaler()
    scaled_array = scaler.fit_transform(log_df[SPEND_COLS])
    scaled_df = pd.DataFrame(scaled_array, columns=SPEND_COLS, index=log_df.index)

    print("Applied StandardScaler: x_scaled = (x - mean) / std, fit on log-transformed data.")
    print("\nPost-scaling summary statistics (expect mean ~0, std ~1 per column):")
    print(scaled_df.describe().round(2).to_string())

    means = scaled_df.mean().round(6)
    stds = scaled_df.std().round(3)
    print(f"\nMeans (should be ~0): {means.to_dict()}")
    print(f"Std devs (should be ~1): {stds.to_dict()}")
    print("[OK] Standardization verified — all features now on a comparable scale for K-Means.")
    print("[NOTE] Log1p reduces skewness substantially; perfect symmetry is not required for K-Means.")

    return scaled_df, scaler

## SECTION 6: Visualize Final Scaled Feature Ranges

In [ ]:
# SECTION 6: Visualize Final Scaled Feature Ranges
# ===========================================================================
def plot_scaled_boxplot(scaled_df: pd.DataFrame) -> None:
    """Single combined boxplot confirming all 6 features now share a
    comparable scale after log1p + standardization."""
    print("\n" + "=" * 70)
    print("SECTION 6: VISUALIZE FINAL SCALED FEATURE RANGES")
    print("=" * 70)

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(data=scaled_df, ax=ax, palette="Set2")
    ax.set_title("Standardized Features (log1p + StandardScaler) — Comparable Scale Check")
    ax.set_ylabel("Standardized value (z-score)")
    ax.axhline(0, color="gray", linestyle="--", linewidth=1)
    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/08_scaled_features_boxplot.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {FIG_DIR}/08_scaled_features_boxplot.png")

## SECTION 7: Persist Transformed Data for Downstream Phases

In [ ]:
# SECTION 7: Persist Transformed Data for Downstream Phases
# ===========================================================================
def save_transformed_data(scaled_df: pd.DataFrame) -> None:
    """Save the final scaled feature matrix to disk so Phase 5+ can load it
    directly without re-running this pipeline."""
    print("\n" + "=" * 70)
    print("SECTION 7: PERSIST TRANSFORMED DATA")
    print("=" * 70)

    out_path = "scaled_features.csv"
    scaled_df.to_csv(out_path, index=False)
    print(f"[OK] Saved scaled, log-transformed feature matrix to '{out_path}' "
          f"(shape: {scaled_df.shape}). This is the input for Phase 5 "
          "(algorithm selection & K determination).")

## MAIN — run Phase 4 end to end

In [ ]:
# MAIN — run Phase 4 end to end
# ===========================================================================
if __name__ == "__main__":
    features_df = get_feature_matrix()
    log_df = apply_log_transform(features_df)
    skew_comparison = verify_skew_reduction(features_df, log_df)
    plot_before_after_histograms(features_df, log_df)
    scaled_df, scaler = standardize_features(log_df)
    plot_scaled_boxplot(scaled_df)
    save_transformed_data(scaled_df)

    print("\n" + "=" * 70)
    print("PHASE 4 COMPLETE")
    print("=" * 70)
    print("[OK] log1p transform applied — skewness substantially reduced (see Section 3).")
    print("[OK] StandardScaler applied — all 6 features now zero mean, unit variance.")
    print("[OK] Transformed data saved to scaled_features.csv.")
    print("[OK] Ready for Phase 5 (Algorithm Selection & K Determination).")

### Phase 4 checkpoint

Review the outputs and figures generated by this phase before moving to the next phase.